# chains.py — User Guide

This notebook walks through the `chains.py` module of the `financialtools` package.

## Module Map

**Public API — one function:**

```python
get_stock_evaluation_report(
    ticker: str,
    year: int | None = None,
    base_dir: str = "financial_data",
    sector_file: str = "financialtools/data/sector_ticker.txt",
) -> StockRegimeAssessment
```

**What it returns:** a `StockRegimeAssessment` Pydantic model — a validated, structured LLM output containing the stock's fundamental regime, valuation classification, metric trends, and market comparison.

**`base_dir`** — directory containing all pipeline output Excel files. Defaults to
`"financial_data"` (relative to CWD). External consumers pass their own absolute path.

**`sector_file`** — path to the tab-separated ticker→sector mapping file
(`ticker`, `sector`, `name`, `marginabile`). External consumers pass the path to their
own copy; the file only needs rows for tickers they will actually analyse.

**Hard dependencies before calling:**

| Dependency | Detail |
|---|---|
| `.env` file | Must contain `OPENAI_API_KEY` — loaded automatically via `python-dotenv` |
| `{base_dir}/<TICKER>_*.xlsx` | Produced by `FundamentalEvaluator.evaluate()` — must exist before calling |
| `{base_dir}/metrics_by_sectors.xlsx` | Sector benchmark file for financial metrics |
| `{base_dir}/eval_metrics_by_sectors.xlsx` | Sector benchmark file for evaluation metrics |
| `sector_file` | Maps each ticker to its sector — ticker must be present |

## Section 2 — StockRegimeAssessment Model

The `StockRegimeAssessment` Pydantic model is the structured output returned by the LangChain pipeline. Every field is validated before the object is returned to the caller.

| Field | Type | Description |
|---|---|---|
| `ticker` | `str` | The ticker symbol of the stock under analysis |
| `regime` | `Literal["bull", "bear"]` | Fundamental regime classification — `bull` = strong/improving, `bear` = weak/deteriorating |
| `regime_rationale` | `str` | Concise explanation justifying the regime, drawing on metrics, composite score, and red flags |
| `metrics_movement` | `str` | Narrative of how key financial metrics moved across years (e.g., `GrossMargin increased, DebtToEquity rose sharply`) |
| `non_aligned_findings` | `Optional[str]` | Contradictory signals or anomalies that do not fit the dominant trend — `None` if none exist |
| `evaluation` | `Literal["overvalued", "undervalued", "fair"]` | Valuation classification based on evaluation metrics (P/E, P/B, P/FCF, EarningsYield, etc.) |
| `evaluation_rationale` | `str` | Concise explanation justifying the valuation classification |
| `market_comparison` | `str` | How the stock's metrics compare to sector peer benchmarks in both fundamentals and valuation |

In [ ]:
from financialtools.pydantic_models import StockRegimeAssessment
import inspect
print(inspect.getsource(StockRegimeAssessment))

## Section 3 — Imports + API Key Setup

The function makes live calls to OpenAI's API. Verify the key is loaded before running any cells that invoke the chain.

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

# Verify key is set before making any API calls
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise EnvironmentError("OPENAI_API_KEY not set — add it to your .env file")
print("API key loaded:", api_key[:8] + "...")

## Section 4 — Imports

In [ ]:
from financialtools.chains import get_stock_evaluation_report
from financialtools.prompts import (
    build_prompt,
    system_prompt_StockRegimeAssessment,
    system_prompt_StockRegimeAssessment_sector,
    system_prompt_noredflags_StockRegimeAssessment,
    system_prompt,
)

## Section 5 — LangChain Pipeline Internals

When `get_stock_evaluation_report(ticker, year, base_dir, sector_file)` is called, the following steps execute in order:

1. **`read_financial_results(ticker, input_dir=base_dir, sheet_name, time?)`** — loads four DataFrames from `{base_dir}/<TICKER>_*.xlsx`: `metrics`, `eval_metrics`, `composite_scores`, `red_flags`. If `year` is passed it is forwarded as `time` to filter to that year.

2. **`get_sector_for_ticker(ticker, sector_file=sector_file)`** — reads the tab-separated mapping file (via Polars) and returns the sector string. Raises `SectorNotFoundError` if the ticker is absent. Pass a custom `sector_file` path for external consumers.

3. **`get_market_metrics(file_path, sector)`** — reads the sector benchmark Excel files and returns a filtered DataFrame of peer-average values. Called twice: once for `{base_dir}/metrics_by_sectors.xlsx` and once for `{base_dir}/eval_metrics_by_sectors.xlsx`.

4. **`dataframe_to_json(df)`** — converts each of the six DataFrames to a JSON string so they can be embedded directly in the prompt.

5. **`ChatPromptTemplate.from_messages([...])`** — assembles a two-message prompt: a system message (the selected prompt constant) and a human message that inlines all six JSON payloads under labelled keys (`Metrics`, `Scores`, `Evaluation Metrics`, `RedFlags`, `Market metrics`, `Market evaluation metrics`).

6. **`ChatOpenAI(model="gpt-4.1-nano", temperature=0)`** — the LLM. Temperature is fixed at 0 for deterministic output.

7. **`PydanticOutputParser(pydantic_object=StockRegimeAssessment)`** wrapped in **`OutputFixingParser.from_llm(...)`** — attempts to parse the LLM response into a `StockRegimeAssessment`. If the primary parse fails, `OutputFixingParser` makes a second LLM call with the malformed output, asking the model to correct it.

8. **Returns `StockRegimeAssessment`** — the fully validated Pydantic object.

**Known limitation (TODO in source):** `system_prompt_StockRegimeAssessment` has no `{format_instructions}` placeholder. The `.format(format_instructions=...)` call in the chain is therefore a no-op — format instructions are silently discarded and never reach the LLM. The `OutputFixingParser` is the primary (and only) recovery path for malformed output. If reliability of structured output is critical, add a `{format_instructions}` placeholder to `build_prompt()` in `prompts.py` and validate LLM output behavior before shipping the change.

## Section 6 — get_stock_evaluation_report

> **LIVE cell** — requires `OPENAI_API_KEY` in `.env` and `financial_data/AAPL_*.xlsx` to exist. Run the download + evaluate pipeline first if the files are missing.

In [ ]:
# LIVE — requires OPENAI_API_KEY and financial_data/AAPL_*.xlsx to exist
ticker = "AAPL"
year = 2023

report = get_stock_evaluation_report(ticker, year=year)
print(f"Ticker:  {report.ticker}")
print(f"Regime:  {report.regime}")
print(f"Evaluation: {report.evaluation}")
print()
print("Regime rationale:")
print(report.regime_rationale)
print()
print("Metrics movement:")
print(report.metrics_movement)

## Section 7 — Inspecting StockRegimeAssessment

In [ ]:
# Show all fields of the report as a dict
report_dict = report.model_dump()
for field, value in report_dict.items():
    print(f"{field}: {str(value)[:120]}")

## Section 8 — Prompts: build_prompt()

`prompts.py` exposes four prompt variants, all built from shared metric-definition blocks (`_FINANCIAL_METRICS_BLOCK`, `_EVAL_METRICS_BLOCK`, `_COMPOSITE_SCORE_BLOCK`, `_RED_FLAGS_BLOCK`). Editing a metric definition in one block automatically propagates to every variant that uses it.

The factory `build_prompt(sector_aware, include_red_flags)` drives three of the four variants:

| Constant | `sector_aware` | `include_red_flags` | Use case |
|---|---|---|---|
| `system_prompt_StockRegimeAssessment` | `False` | `True` | Default — used by `chains.py` |
| `system_prompt_StockRegimeAssessment_sector` | `True` | `True` | Adds sector-comparison instruction |
| `system_prompt_noredflags_StockRegimeAssessment` | `False` | `False` | Omits red flags from data description and prompt |
| `system_prompt` | N/A | N/A | Rich variant: profile section, peer benchmarks, emoji section headers — kept as a literal string because its structure diverges from the factory |

In [ ]:
# Base prompt — with red flags, no sector comparison
p1 = build_prompt(sector_aware=False, include_red_flags=True)
print(f"Base prompt length: {len(p1)} chars")

# Sector-aware prompt
p2 = build_prompt(sector_aware=True, include_red_flags=True)
print(f"Sector-aware: {len(p2)} chars")

# No red flags
p3 = build_prompt(sector_aware=False, include_red_flags=False)
print(f"No red flags: {len(p3)} chars")

# Diff between base and sector-aware
diff_lines = [l for l in p2.splitlines() if l not in p1.splitlines()]
print(f"\nLines added by sector_aware=True: {diff_lines}")

## Section 9 — Changing the Prompt

`chains.py` imports `system_prompt_StockRegimeAssessment` at **module load time** (top-level import). This means:

- Reassigning the module-level name after import has no effect on an already-imported `chains` module.
- To use a different prompt, the chain must be rebuilt — either by modifying `chains.py` directly or by calling the chain internals directly in your own code (copy the chain construction logic and substitute the desired prompt string).

The cell below shows how to inspect the sector-aware prompt before deciding whether to use it.

In [ ]:
# To use the sector-aware prompt, pass it directly to the chain builder
# (requires modifying chains.py or calling internals directly — show the pattern)
from financialtools.prompts import system_prompt_StockRegimeAssessment_sector
print("Sector-aware prompt preview:")
print(system_prompt_StockRegimeAssessment_sector[:300])

## Section 10 — Error Handling

The four error scenarios you are most likely to encounter:

**Missing `.env` / unset key**
LangChain raises an `openai.AuthenticationError` (or a plain `ValueError`) when the API key is absent. The explicit check in Section 3 surfaces this before any network call is made.

**Missing `financial_data/*.xlsx`**
`read_financial_results()` in `wrappers.py` uses `pd.read_excel()` internally. If the expected file does not exist, Python raises `FileNotFoundError`. Run `DownloaderWrapper.download_data()` followed by `FundamentalEvaluator.evaluate()` to produce the required files.

**Ticker not in `sector_ticker.txt`**
`get_sector_for_ticker()` raises `SectorNotFoundError` (a subclass of both `FinancialToolsError` and `ValueError`) if the ticker is absent from the lookup file. Add the ticker + sector pair to `financialtools/data/sector_ticker.txt`.

**LLM parse failure**
If the raw LLM response cannot be parsed into `StockRegimeAssessment`, `OutputFixingParser` makes a second LLM call with the malformed output, asking the model to correct it. If the second attempt also fails, a `langchain` output parser exception is raised.

In [ ]:
from financialtools.exceptions import SectorNotFoundError
try:
    get_stock_evaluation_report("INVALID_TICKER_XYZ")
except SectorNotFoundError as e:
    print(f"SectorNotFoundError: {e}")
except Exception as e:
    print(f"{type(e).__name__}: {e}")

## Section 11 — End-to-End Prerequisites Checklist

Before calling `get_stock_evaluation_report()`, confirm each of the following:

1. `.env` file exists in the working directory (or any parent) and contains `OPENAI_API_KEY=sk-...`
2. `financialtools/data/sector_ticker.txt` contains an entry for the target ticker with its sector
3. `financial_data/<TICKER>_*.xlsx` files exist — produced by running `DownloaderWrapper.download_data()` followed by `FundamentalEvaluator.evaluate()` for the target ticker
4. `financial_data/metrics_by_sectors.xlsx` exists — sector benchmark file for financial metrics (peer averages)
5. `financial_data/eval_metrics_by_sectors.xlsx` exists — sector benchmark file for evaluation metrics (peer averages)

## Section 12 — Common Failure Modes

| Symptom | Cause | Fix |
|---|---|---|
| `EnvironmentError: OPENAI_API_KEY not set` | `.env` missing or key not present | Create `.env` with `OPENAI_API_KEY=sk-...` in the project root |
| `openai.AuthenticationError` | Key set but invalid or revoked | Regenerate API key at platform.openai.com and update `.env` |
| `FileNotFoundError` on `read_financial_results` | `financial_data/<TICKER>_*.xlsx` does not exist | Run `DownloaderWrapper` + `FundamentalEvaluator` for the ticker first |
| `SectorNotFoundError: Ticker 'XYZ' not found` | Ticker absent from `sector_ticker.txt` | Add `XYZ,<sector>` (or the file's expected format) to `financialtools/data/sector_ticker.txt` |
| `SectorNotFoundError: No data found for sector '...'` | Sector present for ticker but absent from benchmark file | Populate `financial_data/metrics_by_sectors.xlsx` with the missing sector's data |
| LangChain output parser exception after two LLM calls | LLM returned non-JSON or fields violate Pydantic constraints even after `OutputFixingParser` retry | Check for model availability issues; consider switching to a more capable model; verify prompt is not truncated |
| `report.non_aligned_findings` is `None` | No contradictory signals detected — this is normal | No action needed; the field is `Optional[str]` |
| `KeyError` / wrong year in results | Passed `year` that has no data in the Excel file | Verify available years in `financial_data/<TICKER>_*.xlsx` before passing `year` |